In [4]:
import pandas as pd
import numpy as np
from time import perf_counter
from datasets import load_dataset
from memory_profiler import memory_usage
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModel

In [5]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRETRAINED_MODEL_NAME = 'answerdotai/ModernBERT-base'
MAX_LEN = 128
MAX_EPOCHS = 4  # Maximum epochs for early stopping
PATIENCE = 3     # Patience for early stopping

tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)

print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
ds = load_dataset("dair-ai/emotion", "split")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,text,label
8756,ive made it through a week i just feel beaten ...,0
4660,i feel this strategy is worthwhile,1
6095,i feel so worthless and weak what does he have...,0
304,i feel clever nov,1
8241,im moved in ive been feeling kind of gloomy,0
...,...,...
3919,i didnt expected to be that much addicted to t...,3
13686,i drove to class i was feeling a little appreh...,4
11396,i am feeling quite impressed with myself becau...,5
1534,i feel some control over caring for the little...,2


In [7]:
class MultiClassClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels  # Labels should be integers: 0, 1, 2, ..., num_classes-1
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)  # Changed to scalar tensor of type long
        }

In [8]:
class ModernBertForMultiClassClassification(nn.Module):
    def __init__(self, num_classes):
        super(ModernBertForMultiClassClassification, self).__init__()
        self.bert = AutoModel.from_pretrained(PRETRAINED_MODEL_NAME)
        self.pre_classifier = nn.Linear(768, 768)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_classes)  # Output size is num_classes
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_state = outputs[0][:, 0]  # CLS token
        pooled_output = self.pre_classifier(hidden_state)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits  # Return raw logits, no sigmoid

In [9]:
def get_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precisions, recalls, f1s, supports = precision_recall_fscore_support(y_true, y_pred)

    return acc, precisions, recalls, f1s

In [10]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, save_path, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    start_train = perf_counter()
    
    # Initialize best metrics
    best_train_acc = 0
    best_train_precisions = None
    best_train_recalls = None
    best_train_f1s = None
    best_val_acc = 0
    best_val_precisions = None
    best_val_recalls = None
    best_val_f1s = None
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_true = []
        
        for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{max_epochs}', leave=False):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size,)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # Shape: (batch_size, num_classes)
            loss = criterion(outputs, labels)  # CrossEntropyLoss expects logits and long labels
            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1).cpu().numpy()  # Get class indices
            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            loss.backward()
            optimizer.step()
        
        train_loss /= len(train_dataloader)
        train_preds = np.array(train_preds)
        train_true = np.array(train_true)
        train_acc, train_precisions, train_recalls, train_f1s = get_metrics(train_true, train_preds)
        
        model.eval()
        val_loss = 0
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in val_dataloader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
        
        val_loss /= len(val_dataloader)
        val_preds = np.array(val_preds)
        val_true = np.array(val_true)
        val_acc, val_precisions, val_recalls, val_f1s = get_metrics(val_true, val_preds)
        
        print(f"Epoch {epoch + 1}/{max_epochs} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1s}")
        print(f"Epoch {epoch + 1}/{max_epochs} - Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1s}")
        
        # Early stopping logic
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_train_acc = train_acc
            best_train_precisions = train_precisions
            best_train_recalls = train_recalls
            best_train_f1s = train_f1s
            best_val_acc = val_acc
            best_val_precisions = val_precisions
            best_val_recalls = val_recalls
            best_val_f1s = val_f1s
            torch.save(model.state_dict(), save_path)
            epochs_no_improve = 0
            print("Model saved!")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break
    
    total_train_time = perf_counter() - start_train
    return (best_train_acc, best_train_precisions, best_train_recalls, best_train_f1s,
            best_val_acc, best_val_precisions, best_val_recalls, best_val_f1s, total_train_time)

In [11]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions = []
    true_labels = []
    classification_times = []
    
    start_test = perf_counter()
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            for i in range(input_ids.size(0)):
                input_id = input_ids[i].unsqueeze(0)
                attention_mask_sample = attention_mask[i].unsqueeze(0)
                label = labels[i].item()
                
                start_time = perf_counter()
                
                output = model(input_ids=input_id, attention_mask=attention_mask_sample)  # Shape: (1, num_classes)
                pred = torch.argmax(output, dim=1).item()  # Scalar integer
                
                predictions.append(pred)
                true_labels.append(label)
                classification_times.append(perf_counter() - start_time)
    
    total_test_time = perf_counter() - start_test
    print(f"Test Time: {total_test_time:.2f} seconds")
    
    predictions = np.array(predictions)
    true_labels = np.array(true_labels)
    
    acc, precisions, recalls, f1s = get_metrics(true_labels, predictions)
    
    print("Test Metrics:")
    print("Accuracy:", acc)
    print("F1s:", f1s)
    print("Precisions:", precisions)
    print("Recalls:", recalls)
    
    return predictions, true_labels

In [ ]:
train_texts = train_df['text'].values
train_labels = train_df['label'].values  # Must be integers: 0, 1, 2, ..., num_classes-1

val_texts = val_df['text'].values
val_labels = val_df['label'].values

test_texts = test_df['text'].values
test_labels = test_df['label'].values

train_dataset = MultiClassClassificationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = MultiClassClassificationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = MultiClassClassificationDataset(test_texts, test_labels, tokenizer, MAX_LEN)

seeds = [2, 3, 5]
batch_sizes = [16, 32]
learning_rates = [5e-5, 3e-5, 2e-5]
results = []

# Get number of classes from the training data
num_classes = train_df['label'].nunique()

# Grid search loop
for batch_size in batch_sizes:
    for learning_rate in learning_rates:
        for seed in seeds:
            torch.manual_seed(seed)
            model = ModernBertForMultiClassClassification(num_classes).to(DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
            criterion = nn.CrossEntropyLoss()
            
            train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
            val_dataloader = DataLoader(val_dataset, batch_size=batch_size)
            test_dataloader = DataLoader(test_dataset, batch_size=batch_size)
            
            save_path = f'results/modernbert_multiclass3_bs{batch_size}_lr{learning_rate}_seed{seed}.pt'

            # Train
            if torch.cuda.is_available():
                torch.cuda.reset_max_memory_allocated()

            max_memory_usage_train, retval = memory_usage(
                (train_model, (model, train_dataloader, val_dataloader, optimizer, criterion, save_path),
                 {'max_epochs': MAX_EPOCHS, 'patience': PATIENCE}), max_usage=True, retval=True)
            
            max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

            (train_acc, train_precisions, train_recalls, train_f1s,
             val_acc, val_precisions, val_recalls, val_f1s, total_train_time) = retval
            
            # Load best model
            model.load_state_dict(torch.load(save_path))
            
            # Evaluate
            if torch.cuda.is_available():
                torch.cuda.reset_max_memory_allocated()

            start = perf_counter()
            max_memory_usage_test, test_retval = memory_usage(
                (evaluate_model, (model, test_dataloader), {}), max_usage=True, retval=True)
            total_test_time = perf_counter() - start

            max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

            predictions, true_labels = test_retval
            test_acc, test_precisions, test_recalls, test_f1s = get_metrics(true_labels, predictions)
            
            # Store results for each individual seed
            results.append({
                'seed': seed,
                'batch_size': batch_size,
                'learning_rate': learning_rate,
                'train_acc': train_acc,
                'train_precisions': train_precisions.tolist(),
                'train_recalls': train_recalls.tolist(),
                'train_f1s': train_f1s.tolist(),
                'max_memory_usage_train': max_memory_usage_train,
                'max_vram_usage_train': max_vram_usage_train,
                'total_train_time': total_train_time,
                'val_acc': val_acc,
                'val_precisions': val_precisions.tolist(),
                'val_recalls': val_recalls.tolist(),
                'val_f1s': val_f1s.tolist(),
                'test_acc': test_acc,
                'test_precisions': test_precisions.tolist(),
                'test_recalls': test_recalls.tolist(),
                'test_f1s': test_f1s.tolist(),
                'max_memory_usage_test': max_memory_usage_test,
                'max_vram_usage_test': max_vram_usage_test,
                'total_test_time': total_test_time
            })

In [ ]:
df = pd.DataFrame(results)
df.to_csv('results/modernbert_multiclass3.csv', index=False)

In [ ]:
df